In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds

2026-02-23 14:26:47.247772: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-23 14:26:48.569013: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-23 14:26:52.521437: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [11]:
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

In [12]:
def normalize_img(image, label):
  """Normalizes images: `uint8` -> `float32`."""
  return tf.cast(image, tf.float32) / 255., label

ds_train = ds_train.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(128)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

In [13]:
ds_test = ds_test.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(128)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

In [14]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(10)
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

model.fit(
    ds_train,
    epochs=6,
    validation_data=ds_test,
)

Epoch 1/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - loss: 0.3541 - sparse_categorical_accuracy: 0.9031 - val_loss: 0.1873 - val_sparse_categorical_accuracy: 0.9472
Epoch 2/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.1634 - sparse_categorical_accuracy: 0.9535 - val_loss: 0.1338 - val_sparse_categorical_accuracy: 0.9613
Epoch 3/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.1195 - sparse_categorical_accuracy: 0.9656 - val_loss: 0.1214 - val_sparse_categorical_accuracy: 0.9642
Epoch 4/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0943 - sparse_categorical_accuracy: 0.9725 - val_loss: 0.1016 - val_sparse_categorical_accuracy: 0.9699
Epoch 5/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0766 - sparse_categorical_accuracy: 0.9780 - val_loss: 0.0916 - val_sparse_categorical_accuracy: 0.9733
Epoch 6/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0638 - sparse_categorical_accuracy: 0.9812 - val_loss: 0.0865 - val_sparse_categorical_accuracy: 0.9730


In [15]:
# Save the trained model
model.save('mnist_model.keras')